# Train the emergency-vehicle model on REAL-WORLD data (free Colab GPU)

**What was wrong before:** a model trained mostly on close-up photos of emergency vehicles has never seen an ordinary truck or car in a real street, so it guesses *fire truck* for trucks and *police car* for cars.

**What this notebook does differently**
1. Downloads several public emergency-vehicle datasets (Roboflow Universe) and merges them into 3 classes.
2. Adds **real street pictures of ordinary cars, trucks and buses** (COCO) as negatives: pictures that must NOT be flagged. Captions mentioning ambulances, fire engines or police are filtered out.
3. Optionally adds frames from **your own camera footage** as more negatives.
4. Trains, then looks for emergency vehicles hiding among the negatives (so they do not teach the model the wrong thing).
5. Tests on **held-out real pictures** the model never saw and reports how many ordinary vehicles it wrongly flags, and as which class.

**Before you start**
1. *Runtime > Change runtime type > T4 GPU*.
2. Make a free Roboflow account and copy your **API key** (app.roboflow.com > Settings > API Keys).

Total time: about 45 to 90 minutes. Colab sessions are temporary: run the last cell to download the results.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: use Runtime > Change runtime type > T4 GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
BRANCH = 'main'
!git clone --branch $BRANCH https://github.com/aditya182005/Real_Time_AI_Traffic_Monitoring_Violation_Enforcement.git traffic-agent
%cd traffic-agent
!pip -q install roboflow ultralytics "supervision>=0.20,<0.31"

In [ ]:
from getpass import getpass
ROBOFLOW_KEY = getpass('Paste your Roboflow API key: ')

In [ ]:
# (workspace, project, version or None for the newest, force-one-class or None)
# All are public, CC BY 4.0 (check each page before publishing your work).
SOURCES = [
    ('idk-zjlje', 'emergency-vehicles-v59qx', 10, None),
    ('traffic-rbwic', 'emergency-vehicles-snzgj', None, None),
    ('yolo-fn1iu', 'emergency-vehicles-detection-xockh-af7sr', None, None),
    ('vignesh-jqvmq', 'emergency-vehicle-detection-q4kwr', 3, None),
    ('yolo-emergency-recognition', 'ambulance-detection-wdbvs', 1, None),
    ('yolo-emergency-recognition', 'fire-truck-xumw3-jlvpe', 1, 'fire_truck'),
]

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_KEY)
folders = []

for workspace, project_name, version, forced in SOURCES:
    project = rf.workspace(workspace).project(project_name)
    last_error = None

    for number in ([version] if version else range(12, 0, -1)):
        try:
            data = project.version(number).download('yolov8', location=f'downloads/{project_name}')
            break
        except Exception as exc:
            last_error = exc
    else:
        print('SKIPPED', project_name, '->', last_error)
        continue

    folders.append(data.location + (f':{forced}' if forced else ''))
    print('downloaded', project_name, 'v', number)

print(len(folders), 'datasets ready')

## Real-world negatives: ordinary cars, trucks and buses
About half of these pictures contain a truck or a bus, the usual look-alikes of fire trucks and ambulances. 400 are set aside and **never used for training**: they are the honest test at the end.

In [ ]:
!python scripts/fetch_coco_negatives.py --output data/negatives_all --max-images 2400

import glob, os, random, shutil
pictures = sorted(glob.glob('data/negatives_all/*.jpg'))
assert len(pictures) > 500, f'Only {len(pictures)} negatives downloaded: read the output above.'
random.Random(0).shuffle(pictures)
os.makedirs('data/negatives_heldout', exist_ok=True)
os.makedirs('data/negatives_train', exist_ok=True)

for path in pictures[:400]:
    shutil.move(path, 'data/negatives_heldout/')

for path in pictures[400:]:
    shutil.move(path, 'data/negatives_train/')

print(len(os.listdir('data/negatives_train')), 'for training,', len(os.listdir('data/negatives_heldout')), 'held out')

### Optional: add frames from your own camera footage
Upload one or more clips with **no emergency vehicle in them**. Frames taken every 2 seconds become extra negatives that match your road. Skip this cell if you have none.

In [ ]:
USE_OWN_VIDEOS = False   # set to True, run the cell, then pick your files

OWN_NEGATIVES = ''
if USE_OWN_VIDEOS:
    from google.colab import files
    os.makedirs('own_videos', exist_ok=True)
    for name, data in files.upload().items():
        open(f'own_videos/{name}', 'wb').write(data)
    !python scripts/extract_frames.py --video own_videos --output data/negatives_own --every-seconds 2
    OWN_NEGATIVES = '--negatives data/negatives_own'

In [ ]:
# Merge. READ THE OUTPUT: every class of every dataset is shown with the class it
# was mapped to. An UNKNOWN class is ignored.
assert folders, 'No dataset downloaded: check the SKIPPED lines above.'
sources = ' '.join(f'--source "{f}"' for f in folders)
!python scripts/build_emergency_dataset.py $sources --negatives data/negatives_train $OWN_NEGATIVES --max-negatives 1500

**Check the table above.** Each class should have a few hundred boxes in `train` and 30 or more in `test`. The **neg** column (pictures with no emergency vehicle) should be roughly 30 to 50% of each split. Too few negatives is the cause of "every truck is a fire truck".

In [ ]:
# Look at real labelled images before training (wrong boxes ruin a model).
import cv2, matplotlib.pyplot as plt
NAMES = ['ambulance', 'fire_truck', 'police_car']
files_ = sorted(glob.glob('data/emergency_vehicle/images/train/*'))
random.seed(1)
if len(files_) < 50:
    raise SystemExit(f'Only {len(files_)} training images were built. Read the merge cell output above.')
fig, axes = plt.subplots(3, 4, figsize=(16, 11))

for axis, path in zip(axes.flat, random.sample(files_, 12)):
    image = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]
    label = path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'

    for line in open(label).read().split('\n'):
        if line.strip():
            c, x, y, w, h = line.split(); x, y, w, h = (float(v) for v in (x, y, w, h))
            p1 = (int((x - w / 2) * width), int((y - h / 2) * height))
            p2 = (int((x + w / 2) * width), int((y + h / 2) * height))
            cv2.rectangle(image, p1, p2, (255, 0, 0), 3)
            cv2.putText(image, NAMES[int(c)], (p1[0], max(20, p1[1] - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 0), 2)

    axis.imshow(image); axis.axis('off')

plt.tight_layout(); plt.show()

## Round 1: train

In [ ]:
!python scripts/train_emergency.py --data data/emergency_vehicle/data.yaml --epochs 60 --device 0 --batch -1 --report emergency_scores.json

## Look for emergency vehicles hiding among the negatives
Some street photos can still contain an ambulance, fire engine or police car. If one is labelled "no emergency vehicle", it teaches the model to ignore real ones. The model now flags the most suspicious negatives: look at them. Real emergency vehicles go in `EXCLUDE` in the next cell. Ordinary trucks and cars that were flagged are the useful *hard* negatives: keep those.

In [ ]:
from ultralytics import YOLO
model = YOLO('models/emergency_vehicle.pt')
negatives = [p for p in sorted(glob.glob('data/emergency_vehicle/images/train/*'))
             if not open(p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt').read().strip()]
flagged = []

for path, result in zip(negatives, model.predict(negatives, conf=0.4, verbose=False, stream=True)):
    if len(result.boxes):
        flagged.append((path, model.names[int(result.boxes.cls[0])], float(result.boxes.conf[0])))

flagged.sort(key=lambda f: -f[2])
print(len(flagged), 'of', len(negatives), 'negatives flagged at confidence 0.4')
shown = flagged[:24]
fig, axes = plt.subplots(4, 6, figsize=(22, 14))

for number, (axis, (path, name, conf)) in enumerate(zip(axes.flat, shown)):
    axis.imshow(cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB))
    axis.set_title(f'#{number}  {name} {conf:.2f}', fontsize=12); axis.axis('off')

plt.tight_layout(); plt.show()

In [ ]:
# Put the numbers (#) of pictures that REALLY show an ambulance / fire truck / police car here.
# Leave it empty if every picture is just an ordinary vehicle.
EXCLUDE = []

for number in EXCLUDE:
    path = shown[number][0]
    os.remove(path)
    os.remove(path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt')

print('removed', len(EXCLUDE), 'pictures')

## Round 2: train again (only needed if you removed pictures; harmless otherwise)

In [ ]:
if EXCLUDE:
    !python scripts/train_emergency.py --data data/emergency_vehicle/data.yaml --epochs 60 --device 0 --batch -1 --report emergency_scores.json
else:
    print('Nothing removed: round 1 is the final model.')

The scores above are on the **test** split of the merged dataset. Above 0.8 mAP is good. The check below is the one that matters for your problem.

In [ ]:
# Real pictures of ordinary cars, trucks and buses that were NEVER used for training.
!python scripts/check_emergency_model.py --data data/emergency_vehicle/data.yaml --negatives data/negatives_heldout --conf 0.5

**How to read it.** Under *4. Real negatives*:
* **5% or less flagged** -> good enough to try on your own footage.
* **More than 5%** -> look at the "wrongly called" lines (for example `fire_truck`). Run the cell again with a higher `--conf` (0.6, 0.7) to see the trade-off, or add more negatives (`--max-images 4000` in the download cell, or your own video frames) and train again.

A low flagged share with a decent mAP is what an accurate model looks like. A high mAP with many flagged negatives is the problem you reported.

In [ ]:
from google.colab import files
!zip -q results.zip models/emergency_vehicle.pt emergency_scores.json data/emergency_vehicle/dataset_report.json runs/detect/emergency_vehicle/results.png runs/detect/emergency_vehicle/confusion_matrix.png
files.download('models/emergency_vehicle.pt')
files.download('results.zip')

## Then, on your own computer
1. Put `emergency_vehicle.pt` at `models/emergency_vehicle.pt` in the project (replace the old one).
2. Run `python scripts/check_setup.py --full`.
3. Test on your own clips: `python scripts/footage_report.py --normal clips/normal --emergency clips/emergency` (see docs/EVALUATION.md).
4. If ordinary trucks or cars are still flagged on your own footage, extract frames from those clips with `python scripts/extract_frames.py`, add them as negatives, and train again.

Until you have measured false alarms on your own footage, treat emergency detection as a suggestion that a person confirms.